# Food Recommendation and Classification System

This notebook demonstrates the execution of the machine learning pipeline, refactored using SOLID and Object-Oriented Programming (OOP) principles.
We will go through the pipeline step-by-step so you can see the execution stages and statuses.

## 1. Setup & Dependencies

First, we install all required dependencies from `requirements.txt`.

In [ ]:
import subprocess
import sys

print("Checking and installing dependencies from requirements.txt...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--quiet"])
    print("Dependencies installed successfully!\n")
except Exception as e:
    print(f"Failed to install dependencies: {e}")

import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from src.data.data_loader import KaggleDataLoader
from src.data.preprocessor import DataPreprocessor
from src.data.dataset_builder import DatasetBuilder
from src.models.text_models import TfIdfModel, EmbeddingModel
from src.models.vision_models import ResNet50Model, VGG16Model
from src.models.multimodal import MultimodalModel
from src.evaluation.evaluator import ModelEvaluator
from src.evaluation.visualizer import Visualizer
from sklearn.model_selection import train_test_split

## 2. Data Loading

We use the `KaggleDataLoader` to dynamically download and load the raw CSV data.

In [ ]:
loader = KaggleDataLoader("pes12017000148/food-ingredients-and-recipe-dataset-with-images")
loader.download()

csv_file = 'Food Ingredients and Recipe Dataset with Image Name Mapping.csv'
df = loader.load_csv(csv_file)
images_dir = loader.get_images_dir()
print(f"Loaded {len(df)} recipes.")

## 3. Data Preprocessing & Pipeline Building

We use the `DataPreprocessor` to clean text, encode labels, and generate missing values.
Then, we split the data and use `DatasetBuilder` to create memory-efficient `tf.data.Dataset` pipelines.

In [ ]:
preprocessor = DataPreprocessor(df, images_dir)
preprocessor.generate_missing_cuisines()
preprocessor.remove_missing_images()
labels = preprocessor.encode_labels()

num_classes = preprocessor.get_num_classes()
class_names = preprocessor.get_class_names()
print(f"Number of classes: {num_classes}")

X_tfidf, X_padded, tokenizer = preprocessor.preprocess_text_features()

train_idx, test_idx = train_test_split(preprocessor.df.index, test_size=0.2, random_state=42, stratify=labels)
train_df = preprocessor.df.loc[train_idx]
test_df = preprocessor.df.loc[test_idx]
y_train = labels.loc[train_idx].values
y_test = labels.loc[test_idx].values

builder = DatasetBuilder(batch_size=32)

train_resnet = builder.create_image_dataset(train_df, y_train, 'resnet')
test_resnet = builder.create_image_dataset(test_df, y_test, 'resnet')

train_vgg = builder.create_image_dataset(train_df, y_train, 'vgg')
test_vgg = builder.create_image_dataset(test_df, y_test, 'vgg')

train_multi = builder.create_multimodal_dataset(train_df, X_tfidf[train_idx], y_train)
test_multi = builder.create_multimodal_dataset(test_df, X_tfidf[test_idx], y_test)

evaluator = ModelEvaluator(class_names)

## 4. NLP Models

Training the TF-IDF MLP and Embedding MLP.

In [ ]:
print("\n--- Training TF-IDF Model ---")
tfidf_model = TfIdfModel(input_dim=X_tfidf.shape[1], num_classes=num_classes)
history_tfidf = tfidf_model.train(X_tfidf[train_idx], y_train, X_tfidf[test_idx], y_test, epochs=10)
evaluator.evaluate(tfidf_model, X_tfidf[test_idx], y_test, "TF-IDF MLP")
Visualizer.plot_training_history(history_tfidf, "TF-IDF MLP")

In [ ]:
print("\n--- Training Embeddings Model ---")
emb_model = EmbeddingModel(
    vocab_size=len(tokenizer.word_index) + 1, 
    embedding_dim=100, 
    input_length=X_padded.shape[1], 
    num_classes=num_classes
)
history_emb = emb_model.train(X_padded[train_idx], y_train, X_padded[test_idx], y_test, epochs=10)
evaluator.evaluate(emb_model, X_padded[test_idx], y_test, "Embedding MLP")
Visualizer.plot_training_history(history_emb, "Embedding MLP")

## 5. Computer Vision Models

Training ResNet50 and VGG16 Transfer Learning Models.

In [ ]:
print("\n--- Training ResNet50 Model ---")
resnet_model = ResNet50Model(num_classes=num_classes)
history_resnet = resnet_model.train(train_resnet, test_resnet, epochs=5)
evaluator.evaluate(resnet_model, test_resnet, y_test, "ResNet50")
Visualizer.plot_training_history(history_resnet, "ResNet50")

In [ ]:
print("\n--- Training VGG16 Model ---")
vgg_model = VGG16Model(num_classes=num_classes)
history_vgg = vgg_model.train(train_vgg, test_vgg, epochs=5)
evaluator.evaluate(vgg_model, test_vgg, y_test, "VGG16")
Visualizer.plot_training_history(history_vgg, "VGG16")

## 6. Multimodal Model

Fusing the Dense text features and Global Average image features to make a final prediction.

In [ ]:
print("\n--- Training Multimodal Model ---")
multi_model = MultimodalModel(
    text_input_dim=X_tfidf.shape[1], 
    image_input_shape=(224, 224, 3), 
    num_classes=num_classes
)
history_multi = multi_model.train(train_multi, test_multi, epochs=5)
evaluator.evaluate(multi_model, test_multi, y_test, "Multimodal Model")
Visualizer.plot_training_history(history_multi, "Multimodal Model")

## 7. Model Explainability (Grad-CAM)

Let's visualize what the CNN (ResNet50) is focusing on when making predictions.

In [ ]:
import numpy as np

print("\n--- Grad-CAM Visualization ---")
sample_indices = np.random.choice(len(test_df), min(3, len(test_df)), replace=False)
y_pred_resnet = resnet_model.predict(test_resnet)
y_pred_resnet_classes = np.argmax(y_pred_resnet, axis=1)

for idx in sample_indices:
    row = test_df.iloc[idx]
    image_path = row['Image_Path']
    pred_class = y_pred_resnet_classes[idx]
    true_class = row['Cuisine_Label']
    
    print(f"Recipe: {row['Title']}")
    print(f"True Cuisine: {class_names[true_class]}")
    print(f"Predicted Cuisine: {class_names[pred_class]}")
    
    try:
        Visualizer.display_gradcam(
            image_path, 
            resnet_model.model, 
            'resnet50',  # We'll map back to the 'resnet50' base model layer
            pred_class
        )
    except Exception as e:
        print(f"Skipping visualization for this image due to error: {e}")